# SVM Implementation
## Support Vector Machines with the Iris Dataset

In this notebook, we'll implement a complete SVM workflow:
1. Import libraries
2. Load and explore the dataset
3. Visualize the data
4. Split into training and test sets
5. Scale the features
6. Train the SVM model
7. Make predictions
8. Evaluate performance
9. Visualize decision boundaries
10. Compare different kernels
11. Explore the effect of parameter C

---
## 1. Importing Essential Libraries

In [ ]:
# Data manipulation and numerical operations
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Display settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

---
## 2. Loading the Dataset

In [ ]:
# Load the Iris dataset
iris = load_iris()

# Create a DataFrame for easier manipulation
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'})

# Display basic info
print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {iris.feature_names}")
print(f"\nTarget classes: {iris.target_names}")
print(f"\nSamples per class:")
print(df['species_name'].value_counts())

In [ ]:
# Preview the data
df.head(10)

In [ ]:
# Summary statistics
df.describe()

---
## 3. Data Visualization

In [ ]:
# Pairplot to visualize relationships between features
sns.pairplot(df, hue='species_name', palette='husl', diag_kind='kde')
plt.suptitle('Iris Dataset - Pairplot Analysis', y=1.02, fontsize=14)
plt.show()

### Pairplot Analysis

**Key observations:**
- **Setosa** (blue) is clearly separable from the other two species
- **Versicolor** and **Virginica** have some overlap
- **Petal features** (length and width) provide better class separation than sepal features
- This suggests SVM should perform well on this dataset

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df.drop(['species', 'species_name'], axis=1).corr(), 
            annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

---
## 4. Train-Test Split

In [ ]:
# Separate features and target
X = iris.data  # Features
y = iris.target  # Target

# Split the data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Ensures equal class distribution
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution: {np.bincount(y_train)}")
print(f"Test set class distribution: {np.bincount(y_test)}")

---
## 5. Feature Scaling

**Why scale?**
- SVM is sensitive to feature magnitudes
- Features with larger ranges can dominate the decision boundary
- StandardScaler transforms features to have mean=0 and std=1

**Important:** Fit the scaler on training data only, then transform both training and test data.

In [ ]:
# Initialize the scaler
scaler = StandardScaler()

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data (using the same scaler fitted on training data)
X_test_scaled = scaler.transform(X_test)

print("Before scaling:")
print(f"  X_train mean: {X_train.mean(axis=0).round(2)}")
print(f"  X_train std:  {X_train.std(axis=0).round(2)}")

print("\nAfter scaling:")
print(f"  X_train_scaled mean: {X_train_scaled.mean(axis=0).round(2)}")
print(f"  X_train_scaled std:  {X_train_scaled.std(axis=0).round(2)}")

---
## 6. Training the SVM Model

In [ ]:
# Create an SVM classifier with a linear kernel
svm_model = SVC(kernel='linear', random_state=42)

# Train the model on scaled training data
svm_model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print(f"\nModel parameters:")
print(f"  Kernel: {svm_model.kernel}")
print(f"  C (regularization): {svm_model.C}")
print(f"  Number of support vectors: {svm_model.n_support_}")
print(f"  Total support vectors: {sum(svm_model.n_support_)}")

---
## 7. Make Predictions

In [ ]:
# Generate predictions for training and test data
y_train_pred = svm_model.predict(X_train_scaled)
y_test_pred = svm_model.predict(X_test_scaled)

print("Predictions generated!")
print(f"\nSample predictions (first 10 test samples):")
print(f"  Actual:    {y_test[:10]}")
print(f"  Predicted: {y_test_pred[:10]}")

---
## 8. Evaluate the Model

In [ ]:
# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("=" * 50)
print("MODEL PERFORMANCE")
print("=" * 50)
print(f"\nTraining Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy:     {test_accuracy:.2%}")

In [ ]:
# Confusion Matrix - Training Data
print("\n" + "=" * 50)
print("TRAINING DATA - Confusion Matrix")
print("=" * 50)
cm_train = confusion_matrix(y_train, y_train_pred)
print(cm_train)

print("\nClassification Report (Training):")
print(classification_report(y_train, y_train_pred, target_names=iris.target_names))

In [ ]:
# Confusion Matrix - Test Data
print("\n" + "=" * 50)
print("TEST DATA - Confusion Matrix")
print("=" * 50)
cm_test = confusion_matrix(y_test, y_test_pred)
print(cm_test)

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred, target_names=iris.target_names))

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training confusion matrix
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', 
            xticklabels=iris.target_names, yticklabels=iris.target_names, ax=axes[0])
axes[0].set_title('Training Data - Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Test confusion matrix
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', 
            xticklabels=iris.target_names, yticklabels=iris.target_names, ax=axes[1])
axes[1].set_title('Test Data - Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

---
## 9. Visualize Decision Boundaries

To visualize the decision boundary, we'll use only 2 features (sepal length and sepal width).

In [ ]:
def plot_decision_boundary(X, y, model, title, ax):
    """
    Plot decision boundary for a 2D feature space.
    """
    h = 0.02  # Step size in the mesh
    
    # Create mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Make predictions on the mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.5)
    
    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, 
                         edgecolors='black', s=50)
    
    ax.set_title(title)
    ax.set_xlabel('Sepal Length (scaled)')
    ax.set_ylabel('Sepal Width (scaled)')
    
    return scatter

In [ ]:
# Use only the first 2 features for visualization
X_train_2d = X_train_scaled[:, :2]
X_test_2d = X_test_scaled[:, :2]

# Train a new model on 2D data
svm_2d = SVC(kernel='linear', random_state=42)
svm_2d.fit(X_train_2d, y_train)

# Plot decision boundary
fig, ax = plt.subplots(figsize=(10, 7))
plot_decision_boundary(X_train_2d, y_train, svm_2d, 
                       'SVM Decision Boundary (Linear Kernel)', ax)
plt.colorbar(ax.collections[1], label='Class')
plt.show()

print(f"2D Model Accuracy (test): {svm_2d.score(X_test_2d, y_test):.2%}")

---
## 10. Impact of Different SVM Kernels

In [ ]:
# Compare different kernels
kernels = [
    ('linear', 'Linear Kernel'),
    ('poly', 'Polynomial Kernel (degree=3)'),
    ('rbf', 'RBF Kernel')
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (kernel, title) in zip(axes, kernels):
    # Train model with specific kernel
    if kernel == 'poly':
        model = SVC(kernel=kernel, degree=3, random_state=42)
    else:
        model = SVC(kernel=kernel, random_state=42)
    
    model.fit(X_train_2d, y_train)
    
    # Plot decision boundary
    plot_decision_boundary(X_train_2d, y_train, model, title, ax)
    
    # Add accuracy to title
    acc = model.score(X_test_2d, y_test)
    ax.set_title(f'{title}\nTest Accuracy: {acc:.2%}')

plt.tight_layout()
plt.show()

### Kernel Comparison Analysis

| Kernel | Characteristics |
|--------|----------------|
| **Linear** | Straight decision boundary, fast, best for linearly separable data |
| **Polynomial** | Curved boundary, can capture more complex patterns |
| **RBF** | Flexible non-linear boundary, most versatile, good default choice |

---
## 11. Impact of Parameter C on Decision Boundaries

In [ ]:
# Compare different C values
C_values = [0.1, 1, 10, 100]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, C in zip(axes, C_values):
    # Train model with specific C value
    model = SVC(kernel='rbf', C=C, random_state=42)
    model.fit(X_train_2d, y_train)
    
    # Plot decision boundary
    plot_decision_boundary(X_train_2d, y_train, model, f'C = {C}', ax)
    
    # Add accuracy and support vectors count
    acc = model.score(X_test_2d, y_test)
    n_sv = sum(model.n_support_)
    ax.set_title(f'C = {C}\nAccuracy: {acc:.2%} | SVs: {n_sv}')

plt.suptitle('Effect of C Parameter on Decision Boundaries (RBF Kernel)', 
             y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### C Parameter Analysis

| C Value | Effect |
|---------|--------|
| **Low (0.1)** | Wider margin, more misclassifications allowed, better generalization, more support vectors |
| **High (100)** | Tighter margin, fewer misclassifications, risk of overfitting, fewer support vectors |

**Key Insight:** As C increases, the model tries harder to correctly classify all training points, potentially at the cost of generalization.

---
## Summary

In this notebook, we implemented a complete SVM workflow:

1. **Loaded** the Iris dataset (150 samples, 4 features, 3 classes)
2. **Visualized** feature relationships using pairplots
3. **Split** data into 80% training and 20% test sets
4. **Scaled** features using StandardScaler
5. **Trained** an SVM model with a linear kernel
6. **Evaluated** performance using accuracy, confusion matrix, and classification report
7. **Visualized** decision boundaries
8. **Compared** different kernels (linear, polynomial, RBF)
9. **Explored** the effect of the C parameter

### Key Takeaways

- **Always scale your features** before training an SVM
- **Start with RBF kernel** if unsure — it's the most flexible
- **Tune C and gamma** using cross-validation for best results
- **Visualize decision boundaries** to understand model behavior

In [ ]:
print("\n" + "=" * 50)
print("SVM IMPLEMENTATION COMPLETE!")
print("=" * 50)